# D3PM training on Colab Pro

Trains discrete D3PM on malware opcode sequences (zeroaccess + zbot + winwebsec).
Outputs land in your Google Drive copy of the project, so they're available on your
MacBook for evaluation/paper writing.

**Before you start:** make sure `Runtime → Change runtime type → Hardware accelerator = GPU`
(T4 is fine; A100 if available is ~3× faster).

**Total expected wall-clock on T4:** ~2–3 hours for all 3 families. Stay on the tab
(or run [keep-awake](https://chromewebstore.google.com/) on it) so Colab doesn't idle-disconnect.

## 1. Mount Drive + cd into project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust this path if you uploaded to a different folder.
PROJECT_DIR = '/content/drive/MyDrive/diffusion-proj'
%cd $PROJECT_DIR
!ls

## 2. Install dependencies + verify GPU

Re-run this cell every fresh Colab session (~1 min). Colab already has torch+CUDA
preinstalled; we only need gensim + a couple of others.

In [ ]:
!pip install -q gensim==4.4.0 tqdm

In [ ]:
import torch
print(f'torch:       {torch.__version__}')
print(f'CUDA avail:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU:         {p.name}')
    print(f'GPU memory:  {p.total_memory / 1e9:.1f} GB')

## 3. (Optional) Clear stale D3PM artifacts

Your previous (broken) run left behind `d3pm_w2v.pkl`, `d3pm_real_embeddings.npy`,
and the old synth files for zeroaccess. The W2V is technically fine to reuse, but
for a clean slate, run this cell. **Skip if you want to compare against the old W2V.**

In [ ]:
!rm -fv checkpoints/zeroaccess/d3pm.pt checkpoints/zeroaccess/d3pm_vocab.pkl checkpoints/zeroaccess/d3pm_w2v.pkl checkpoints/zeroaccess/d3pm_real_embeddings.npy checkpoints/zeroaccess/d3pm_losses.json
!rm -rfv synthetic/zeroaccess/d3pm.npy synthetic/zeroaccess/d3pm_sequences

## 4. Smoke test (~5–10 min on T4)

Trains on 100 zeroaccess files for 30 epochs at max_len=1024. Confirms the W2V-fix
pipeline produces sane numbers before you commit to the long full run.

**Pass criteria:**
- Cosine deviation < 0.30 (baseline DDPM is ~0.01; old broken D3PM was 0.72)
- Binary classification F1 < 0.95 (closer to 0.5 is better)

If those numbers are way off, **stop and debug** before kicking off the full run.

In [ ]:
!python -m mdiff.train --variant d3pm --families zeroaccess \
    --max-files 100 --epochs 30 --max-len 1024 --batch 8

In [ ]:
!python -m mdiff.generate --variant d3pm --family zeroaccess \
    --n 50 --max-len 1024 --batch 32

In [ ]:
!python -m mdiff.evaluate --variant d3pm --families zeroaccess \
    --compare-against eval_results/ddpm_full_report.json

### Decision point

Read the printed cosine deviation + classifier F1 above. If they look reasonable,
continue to section 5. If they're still broken, post the output and stop here.

## 5. Full run — zeroaccess (~25–40 min on T4)

100 epochs, max_len=2048, all 1311 files. The smoke test artifacts get overwritten.

In [ ]:
# Wipe smoke-test artifacts so the full run starts fresh.
!rm -fv checkpoints/zeroaccess/d3pm.pt checkpoints/zeroaccess/d3pm_vocab.pkl checkpoints/zeroaccess/d3pm_w2v.pkl checkpoints/zeroaccess/d3pm_real_embeddings.npy

In [ ]:
!python -m mdiff.train --variant d3pm --families zeroaccess \
    --epochs 100 --max-len 2048 --batch 8

In [ ]:
!python -m mdiff.generate --variant d3pm --family zeroaccess \
    --n 200 --max-len 2048 --batch 32

## 6. Full run — zbot (~30–50 min on T4)

2136 files. Same hyperparameters.

In [ ]:
!python -m mdiff.train --variant d3pm --families zbot \
    --epochs 100 --max-len 2048 --batch 8

In [ ]:
!python -m mdiff.generate --variant d3pm --family zbot \
    --n 200 --max-len 2048 --batch 32

## 7. Full run — winwebsec (~40–70 min on T4)

4360 files. Largest family.

In [ ]:
!python -m mdiff.train --variant d3pm --families winwebsec \
    --epochs 100 --max-len 2048 --batch 8

In [ ]:
!python -m mdiff.generate --variant d3pm --family winwebsec \
    --n 200 --max-len 2048 --batch 32

## 8. Multi-family embedding eval (3 families together)

Produces `eval_results/d3pm_full_report.json` with all 6 paper metrics, side-by-side
with the saved DDPM baseline. This is the table you'll put in the paper.

In [ ]:
!python -m mdiff.evaluate --variant d3pm \
    --families zeroaccess zbot winwebsec \
    --compare-against eval_results/ddpm_full_report.json

## 9. Sequence-level eval (per family)

n-gram, edit distance, opcode-frequency KL — D3PM-specific qualitative metrics.

In [ ]:
!python -m mdiff.seq_evaluate --family zeroaccess
!python -m mdiff.seq_evaluate --family zbot
!python -m mdiff.seq_evaluate --family winwebsec

## 10. Sanity check — confirm artifacts wrote back to Drive

If anything's missing here, Drive sync didn't finish — wait 30 s and re-list.

In [ ]:
!ls -la checkpoints/zeroaccess/ checkpoints/zbot/ checkpoints/winwebsec/
!echo '---'
!ls -la synthetic/zeroaccess/ synthetic/zbot/ synthetic/winwebsec/
!echo '---'
!ls -la eval_results/